## Datasets Pipeline

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
import torch
import torch.nn as nn
import torch.nn.functional as F
import subprocess
import sys



#synthetic data
def generate_population_nd(n_samples: int, w_star: np.ndarray, noise_std: float = 0.1):
    dim = len(w_star)
    X = np.random.randn(n_samples, dim)

    # Ensure exact normalization so budget means the same thing
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    logits = np.dot(X, w_star) + np.random.normal(0, noise_std, n_samples)
    f_star_scores = 1 / (1 + np.exp(-logits))
    return X, f_star_scores

# correlated non-linear data ---
def generate_correlated_nonlinear_population(n_samples: int, rho: float, dim: int = 2):
    """Generates data where features have a correlation of rho, with a non-linear boundary."""
    # Build covariance matrix
    cov = np.eye(dim)
    for i in range(dim):
        for j in range(dim):
            if i != j:
                cov[i, j] = rho

    # Cholesky decomposition to induce correlation
    L = np.linalg.cholesky(cov)
    X = np.random.randn(n_samples, dim) @ L.T

    # Define a ground truth scoring function with a non-linear term
    w_star = np.array([1.0, -0.5] + [0.0] * (dim - 2)) # Slanted base boundary
    w_star = w_star / np.linalg.norm(w_star)

    # Logits include a non-linear interaction (sine wave dependent on feature 0)
    logits = X @ w_star + 0.8 * np.sin(np.pi * X[:, 0]) * X[:, 1]
    f_star_scores = 1 / (1 + np.exp(-logits))
    return X.astype(np.float32), f_star_scores, w_star

# --- Ground Truth PyTorch Wrapper (Non-Linear)
class NonLinearGT(torch.nn.Module):
    def __init__(self, w_star):
        super().__init__()
        self.w = torch.tensor(w_star, dtype=torch.float32)
    def forward(self, x):
        # Must perfectly match the non-linearity in the generator
        logits = torch.matmul(x, self.w) + 0.8 * torch.sin(torch.pi * x[:, 0]) * x[:, 1]
        return torch.sigmoid(logits)

#real-world data
def prepare_adult_data(n_samples=2000, seed=42):
    """
    Fetches and preprocesses the Adult Income dataset.
    Trains a non-linear MLP as the Ground Truth model to generate f* scores.
    """
    print("Fetching Adult Income dataset from OpenML...")
    adult = fetch_openml(name='adult', version=2, as_frame=True, parser='auto')
    df = adult.frame

    # Subsample for faster experimentation if requested
    if n_samples is not None:
        df = df.sample(n=n_samples, random_state=seed)

    X = df.drop('class', axis=1)
    # Target: >50K is 1, <=50K is 0
    y = (df['class'] == '>50K').astype(int).values

    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object', 'category']).columns

    # Robust preprocessing
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    print("Preprocessing features...")
    X_processed = preprocessor.fit_transform(X)

    print("Training Ground Truth MLP Model (f*)...")
    # Highly expressive model to serve as Ground Truth
    gt_model_real = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=seed)
    gt_model_real.fit(X_processed, y)

    # The predicted probabilities represent our f* scores
    f_star_scores = gt_model_real.predict_proba(X_processed)[:, 1]

    # Returning gt_model_real so we can use it in run_unified_experiments
    return X_processed.astype(np.float32), f_star_scores, preprocessor, gt_model_real

def prepare_credit_data(n_samples=None, seed=42):
    """
    Fetches and preprocesses the German Credit dataset.
    Target: 1 (Good Credit), 2 (Bad Credit) -> mapped to 1 (Good), 0 (Bad)
    """
    print("Fetching German Credit dataset...")
    # Using OpenML ID 31 for German Credit
    credit = fetch_openml(data_id=31, as_frame=True, parser='auto')
    df = credit.frame

    if n_samples is not None and n_samples < len(df):
        df = df.sample(n=n_samples, random_state=seed)

    X = df.drop('class', axis=1)
    # Target: 'good' vs 'bad'. Let's make 'good' = 1, 'bad' = 0
    y = (df['class'] == 'good').astype(int).values

    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object', 'category']).columns

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    X_processed = preprocessor.fit_transform(X)

    print("Training Ground Truth MLP Model for German Credit...")
    gt_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=seed)
    gt_model.fit(X_processed, y)
    f_star_scores = gt_model.predict_proba(X_processed)[:, 1]

    return X_processed.astype(np.float32), f_star_scores, preprocessor, gt_model

def prepare_compas_data(n_samples=None, seed=42):
    """
    Fetches and preprocesses the COMPAS dataset.
    Target: two_year_recid. We map it so 1 = No Recidivism (favorable), 0 = Recidivism.
    """
    print("Fetching COMPAS dataset (ProPublica Recidivism)...")
    compas = fetch_openml(name="compas-two-years", as_frame=True, parser='auto')
    df = compas.frame

    if n_samples is not None and n_samples < len(df):
        df = df.sample(n=n_samples, random_state=seed)

    # Target is 'two_year_recid'. We want 1 to be the favorable outcome (no recidivism, which is '0' in the dataset)
    if 'two_year_recid' in df.columns:
        target_col = 'two_year_recid'
        y = (df[target_col] == '0').astype(int).values  # 1 if did NOT recidivate
    else:
        target_col = 'class'
        y = (df[target_col] == 0).astype(int).values

    X = df.drop(target_col, axis=1)

    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object', 'category']).columns

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    X_processed = preprocessor.fit_transform(X)

    print("Training Ground Truth MLP Model for COMPAS...")
    gt_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=seed)
    gt_model.fit(X_processed, y)
    f_star_scores = gt_model.predict_proba(X_processed)[:, 1]

    return X_processed.astype(np.float32), f_star_scores, preprocessor, gt_model

def prepare_gmsc_data(n_samples=2000, seed=42):
    """
    Fetches and preprocesses the Give Me Some Credit dataset.
    Target: SeriousDlqin2yrs -> mapped so 1 = Favorable (0 in original).
    """
    print("Loading Give Me Some Credit...")
    df = pd.read_csv('GiveMeSomeCredit.csv').dropna()

    if n_samples is not None and n_samples < len(df):
        df = df.sample(n=n_samples, random_state=seed)

    # Favorable outcome is 0 (No Serious Delinquency in 2 yrs)
    y = (df['SeriousDlqin2yrs'] == 0).astype(int).values
    X = df.drop(['SeriousDlqin2yrs', 'Unnamed: 0'], axis=1, errors='ignore')

    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object', 'category']).columns

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    X_processed = preprocessor.fit_transform(X)

    print("Training Ground Truth MLP Model for Give Me Some Credit...")
    gt_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=seed)
    gt_model.fit(X_processed, y)
    f_star_scores = gt_model.predict_proba(X_processed)[:, 1]

    return X_processed.astype(np.float32), f_star_scores, preprocessor, gt_model


def prepare_folktables_data(n_samples=2000, seed=42):
    """
    Fetches and preprocesses the Folktables ACSIncome dataset.
    """
    try:
        import folktables
    except ImportError:
        print("Installing folktables...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "folktables"])
        import folktables

    from folktables import ACSDataSource, ACSIncome

    print("Loading Folktables (ACSIncome)...")
    data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
    acs_data = data_source.get_data(states=["CA"], download=True)
    features, label, group = ACSIncome.df_to_numpy(acs_data)

    if n_samples is not None and n_samples < len(features):
        np.random.seed(seed)
        idx = np.random.choice(len(features), n_samples, replace=False)
        features = features[idx]
        label = label[idx]

    y = label.flatten().astype(int)

    # Using a simple StandardScaler directly based on the provided snippet
    preprocessor = StandardScaler()
    X_processed = preprocessor.fit_transform(features)

    print("Training Ground Truth MLP Model for Folktables...")
    gt_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=seed)
    gt_model.fit(X_processed, y)
    f_star_scores = gt_model.predict_proba(X_processed)[:, 1]

    return X_processed.astype(np.float32), f_star_scores, preprocessor, gt_model

In [ ]:
# --- Demonstration ---
"""X_adult, f_star_adult, preprocessor_adult, gt_model_adult = prepare_adult_data(n_samples=1000)
print(f"\nProcessed Data Shape: {X_adult.shape}")
print(f"Sample f* scores (first 5): {f_star_adult[:5]}")
X_credit, f_star_credit, preprocessor_credit, gt_model_credit = prepare_credit_data()
print(f"\nGerman Credit Shape: {X_credit.shape}")

X_compas, f_star_compas, preprocessor_compas, gt_model_compas = prepare_compas_data(n_samples=2000) # Subsample for speed
print(f"COMPAS Shape: {X_compas.shape}")"""

print("Testing Give Me Some Credit...")
X_gmsc, f_star_gmsc, preprocessor_gmsc, gt_model_gmsc = prepare_gmsc_data(n_samples=500)
print(f"GMSC Processed Shape: {X_gmsc.shape}")
print(f"GMSC f* scores (first 5): {f_star_gmsc[:5]}")

print("\nTesting Folktables (ACSIncome)...")
X_folk, f_star_folk, preprocessor_folk, gt_model_folk = prepare_folktables_data(n_samples=500)
print(f"Folktables Processed Shape: {X_folk.shape}")
print(f"Folktables f* scores (first 5): {f_star_folk[:5]}")


## Model Training and Evaluation Pipeline

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from itertools import product
from typing import Tuple

# --- Environment & Helper Functions ---
def apply_recourse_optimized(X_rejected: np.ndarray, w_model: np.ndarray, budget_B: float, lr: float = 0.05, max_steps: int = 500, tol: float = 1e-5) -> np.ndarray:
    """
    Optimizes min CE(f(x'), 1) s.t. ||x - x'||_2 <= budget for each individual, until convergence.
    """
    if len(X_rejected) == 0:
        return X_rejected.copy()

    x_tensor = torch.tensor(X_rejected, dtype=torch.float32)
    w_tensor = torch.tensor(w_model, dtype=torch.float32)
    actions = torch.zeros_like(x_tensor, requires_grad=True)

    optimizer = torch.optim.Adam([actions], lr=lr)
    target = torch.ones(x_tensor.shape[0], dtype=torch.float32)

    prev_loss = float('inf')
    for _ in range(max_steps):
        optimizer.zero_grad()
        x_prime = x_tensor + actions
        logits = torch.matmul(x_prime, w_tensor)
        probs = torch.sigmoid(logits)
        loss = F.binary_cross_entropy(probs, target)

        if abs(prev_loss - loss.item()) < tol:
            break
        prev_loss = loss.item()

        loss.backward()
        optimizer.step()

        # Enforce budget constraint (projection onto L2 ball)
        with torch.no_grad():
            norms = torch.norm(actions, p=2, dim=1, keepdim=True)
            mask = norms > budget_B
            if mask.any():
                actions[mask.expand_as(actions)] = (actions / norms * budget_B)[mask.expand_as(actions)]

    return (x_tensor + actions).detach().numpy()

def get_labels(scores: np.ndarray, alpha: float) -> Tuple[np.ndarray, float]:
    threshold = np.quantile(scores, 1 - alpha)
    return (scores >= threshold).astype(int), threshold

class SklearnGTWrapper(nn.Module):
    """Wraps an sklearn model so it can be used as a ground-truth scoring function in PyTorch algorithms."""
    def __init__(self, sklearn_model):
        super().__init__()
        self.model = sklearn_model

    def forward(self, x):
        x_np = x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.array(x)
        probs = self.model.predict_proba(x_np)[:, 1]
        device = x.device if isinstance(x, torch.Tensor) else 'cpu'
        return torch.tensor(probs, dtype=torch.float32, device=device)

def retrain_model(X_shifted, gt_scoring_fn, alpha, model_class=LogisticRegression, model_kwargs=None):
    if model_kwargs is None:
        model_kwargs = {'penalty': None, 'fit_intercept': False, 'max_iter': 5000}

    f_star_syn = gt_scoring_fn(X_shifted)
    y_syn, _ = get_labels(f_star_syn, alpha)

    clf_w1 = model_class(**model_kwargs)
    if len(np.unique(y_syn)) == 2:
        clf_w1.fit(X_shifted, y_syn)
    else:
        clf_w1.fit(X_shifted, np.random.randint(0, 2, size=len(y_syn)))

    f_w1_scores = clf_w1.predict_proba(X_shifted)[:, 1]
    _, t_1 = get_labels(f_w1_scores, alpha)
    return clf_w1, t_1

def evaluate_policy(X, y_hat_0, theta, budget_B, gt_scoring_fn, alpha, model_class=LogisticRegression, model_kwargs=None):
    X_rejected = X[y_hat_0 == 0]
    if len(X_rejected) == 0:
        return np.nan, np.nan

    X_shifted = X.copy()
    rejected_indices = np.where(y_hat_0 == 0)[0]

    norm = np.linalg.norm(theta)
    if norm > budget_B:
        theta = (theta / norm) * budget_B

    X_shifted[rejected_indices] = X_rejected + theta
    clf_w1, t_1 = retrain_model(X_shifted, gt_scoring_fn, alpha, model_class, model_kwargs)

    X_recoursed = X_rejected + theta
    scores_w1 = clf_w1.predict_proba(X_recoursed)[:, 1]
    validity = np.mean((scores_w1 >= t_1).astype(int))

    return float(validity), t_1

## Algorithm 1

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import sys
import os

class LogisticRegressionPT(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1, bias=False)
    def forward(self, x):
        return torch.sigmoid(self.linear(x)).squeeze()

class PolicyNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc2 = nn.Linear(16, input_dim)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

# ==========================================
# VERSION 1: Using Policy Net directly
# ==========================================

def algorithm_1_policy_net(X, y_hat_0, gt_model, policy_net, max_epochs=200, budget=1.0, alpha=0.5, lr=0.01, lr_retrain=0.5, tol=1e-4, w_star=None):
    """
    Algorithm 1 using a Policy Network to directly propose recourse actions until convergence.
    """
    X_tensor = torch.tensor(X, dtype=torch.float32) if not isinstance(X, torch.Tensor) else X
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    current_model = gt_model

    with torch.no_grad():
        initial_probs = current_model(X_tensor)
        t0 = torch.quantile(initial_probs, 1 - alpha).item()

    rejected_mask = (initial_probs < t0).view(-1)
    X_rejected = X_tensor[rejected_mask]

    if len(X_rejected) == 0:
        return np.zeros(X_tensor.shape[1]), []

    validity_history = []
    prev_loss = float('inf')

    for epoch in range(max_epochs):
        optimizer.zero_grad()

        # Policy net directly outputs actions
        actions_rejected = policy_net(X_rejected)

        # Hard L2 Projection
        norms = torch.norm(actions_rejected, p=2, dim=1, keepdim=True)
        actions_rejected = torch.where(norms > budget, actions_rejected / (norms + 1e-8) * budget, actions_rejected)

        actions_full = torch.zeros_like(X_tensor)
        actions_full[rejected_mask] = actions_rejected
        X_prime_full = X_tensor + actions_full

        # Simulate Retraining on Exact Shifted Data
        with torch.no_grad():
            probs_gt = gt_model(X_prime_full)
            t_gt = torch.quantile(probs_gt, 1 - alpha)
            y_syn = (probs_gt >= t_gt).float()

        w1_weight = current_model.linear.weight.clone() if hasattr(current_model, 'linear') else torch.zeros((1, X.shape[1]), requires_grad=True)
        v_w1 = torch.zeros_like(w1_weight) # Momentum buffer
        beta = 0.9
        prev_w1_loss = float('inf')
        for _ in range(100):
            logits_syn = torch.matmul(X_prime_full, w1_weight.t()).squeeze()
            probs_syn = torch.sigmoid(logits_syn)
            loss_w1 = F.binary_cross_entropy(probs_syn, y_syn)
            if abs(prev_w1_loss - loss_w1.item()) < tol:
                break
            prev_w1_loss = loss_w1.item()

            grad_w1 = torch.autograd.grad(loss_w1, w1_weight, create_graph=True)[0]
            v_w1 = beta * v_w1 + (1 - beta) * grad_w1 # Manual Momentum Update
            w1_weight = w1_weight - lr_retrain * v_w1

        logits_full_retrained = torch.matmul(X_prime_full, w1_weight.t()).squeeze()
        probs_full_retrained = torch.sigmoid(logits_full_retrained)
        t1 = torch.quantile(probs_full_retrained, 1 - alpha)

        # Validity Loss
        X_prime_rejected = X_rejected + actions_rejected
        logits_rejected_retrained = torch.matmul(X_prime_rejected, w1_weight.t()).squeeze()
        probs_rejected = torch.sigmoid(logits_rejected_retrained)

        validity_loss = torch.mean(torch.relu(t1 - probs_rejected))
        total_loss = validity_loss

        # Track real validity if w_star is provided
        if w_star is not None:
            mean_theta = np.mean(actions_rejected.detach().numpy(), axis=0)
            norm = np.linalg.norm(mean_theta)
            theta_val = (mean_theta / norm) * budget if norm > 1e-9 else mean_theta
            val, _ = evaluate_policy(X, y_hat_0, theta_val, budget, w_star, alpha)
            validity_history.append(val)

        if tol > 0 and abs(prev_loss - total_loss.item()) < tol:
            break
        prev_loss = total_loss.item()

        total_loss.backward()
        optimizer.step()

    mean_theta = np.mean(actions_rejected.detach().numpy(), axis=0)
    norm = np.linalg.norm(mean_theta)
    theta_out = (mean_theta / norm) * budget if norm > 1e-9 else mean_theta
    return theta_out, validity_history


# ==========================================================
# VERSION 2: Implicit Function Theorem (IFT) Gradients with CG
# ==========================================================

def cg_solve(hvp_fn, b, cg_iters=10, residual_tol=1e-5):
    """
    Conjugate Gradient solver for H * x = b.
    """
    x = torch.zeros_like(b)
    r = b.clone()
    p = b.clone()
    rsold = torch.dot(r.view(-1), r.view(-1))
    for _ in range(cg_iters):
        Ap = hvp_fn(p)
        alpha = rsold / (torch.dot(p.view(-1), Ap.view(-1)) + 1e-8)
        x = x + alpha * p
        r = r - alpha * Ap
        rsnew = torch.dot(r.view(-1), r.view(-1))
        if torch.sqrt(rsnew) < residual_tol:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return x

def compute_ift_w1(X_prime, y_syn, w1_weight, exact_hessian=True):
    """
    Computes the implicit gradient of w1 with respect to the shifted data X_prime.
    Supports both explicit exact Hessian inversion (best for linear models)
    and Conjugate Gradient (best for large neural networks).
    """
    w1 = w1_weight.detach().requires_grad_(True)
    X = X_prime.detach().requires_grad_(True)

    logits = torch.matmul(X, w1.t()).squeeze()
    probs = torch.sigmoid(logits)
    loss = F.binary_cross_entropy(probs, y_syn)

    grad_w1 = torch.autograd.grad(loss, w1, create_graph=True)[0]

    # Mixed derivatives shape: (D, N, D)
    mixed_grads = []
    for g in grad_w1.view(-1):
        mixed_row = torch.autograd.grad(g, X, retain_graph=True)[0]
        mixed_grads.append(mixed_row)
    mixed_grads = torch.stack(mixed_grads) # D x N x D

    if exact_hessian:
        # Exact Hessian matrix formulation and inversion
        H = []
        for g in grad_w1.view(-1):
            H_row = torch.autograd.grad(g, w1, retain_graph=True)[0].view(-1)
            H.append(H_row)
        H = torch.stack(H)

        H_inv = torch.inverse(H + 1e-5 * torch.eye(H.size(0)))
        dw1_dX = -torch.einsum('ij,jkl->ikl', H_inv, mixed_grads)
        return dw1_dX

    else:
        # Conjugate Gradient iterative solver
        def hvp_fn(v):
            grad_v = torch.sum(grad_w1 * v)
            hvp = torch.autograd.grad(grad_v, w1, retain_graph=True)[0]
            return hvp + 1e-5 * v  # Add damping

        dw1_dX = torch.zeros_like(mixed_grads)
        for i in range(mixed_grads.size(0)):
            for j in range(mixed_grads.size(1)):
                b = mixed_grads[i, j]
                v = cg_solve(hvp_fn, b.unsqueeze(0))
                dw1_dX[i, j] = -v.squeeze(0)
        return dw1_dX


class SmoothedThreshold(torch.autograd.Function):
    @staticmethod
    def forward(ctx, probs, alpha, tau):
        t1 = torch.quantile(probs, 1 - alpha)
        ctx.save_for_backward(probs, t1, torch.tensor(tau))
        return t1

    @staticmethod
    def backward(ctx, grad_output):
        probs, t1, tau_tensor = ctx.saved_tensors
        tau = tau_tensor.item()

        diff = (t1 - probs) / tau
        sigma = torch.sigmoid(diff)
        sigma_prime = sigma * (1 - sigma)

        dt1_dp = sigma_prime / (torch.sum(sigma_prime) + 1e-8)
        grad_probs = grad_output * dt1_dp
        return grad_probs, None, None


def algorithm_1_optimized_ce_gen(X, y_hat_0, gt_model, initial_theta, eval_fn=None, max_epochs=200, budget=1.0, alpha=0.5, lr_outer=0.01, lr_inner=0.05, lr_retrain=0.5, max_inner_steps=200, tol=1e-4, smooth_threshold=False, tau=0.01, exact_hessian=True):
    """
    Generalized Algorithm 1 (IFT) that accepts a generic eval_fn for tracking convergence.
    """
    X_tensor = torch.tensor(X, dtype=torch.float32) if not isinstance(X, torch.Tensor) else X
    current_model = gt_model

    theta_param = nn.Parameter(torch.tensor(initial_theta, dtype=torch.float32))
    optimizer = optim.Adam([theta_param], lr=lr_outer)

    with torch.no_grad():
        initial_probs = current_model(X_tensor)
        t0 = torch.quantile(initial_probs, 1 - alpha).item()

    rejected_mask = (initial_probs < t0).view(-1)
    X_rejected = X_tensor[rejected_mask]

    if len(X_rejected) == 0:
        return np.zeros(X_tensor.shape[1]), []

    validity_history = []
    prev_outer_loss = float('inf')

    for epoch in range(max_epochs):
        optimizer.zero_grad()
        actions = torch.zeros_like(X_rejected, requires_grad=True)
        v_a = torch.zeros_like(actions)
        beta = 0.9

        prev_inner_loss = float('inf')
        for _ in range(max_inner_steps):
            x_prime = X_rejected + actions
            logits = torch.matmul(x_prime, theta_param)
            probs = torch.sigmoid(logits)
            loss_inner = F.binary_cross_entropy(probs, torch.ones_like(probs))

            if abs(prev_inner_loss - loss_inner.item()) < tol:
                break
            prev_inner_loss = loss_inner.item()

            grad_a = torch.autograd.grad(loss_inner, actions, create_graph=True)[0]
            v_a = beta * v_a + (1 - beta) * grad_a
            actions = actions - lr_inner * v_a

            norms = torch.norm(actions, p=2, dim=1, keepdim=True)
            actions = torch.where(norms > budget, actions / (norms + 1e-8) * budget, actions)

        actions_full = torch.zeros_like(X_tensor)
        actions_full[rejected_mask] = actions
        X_prime_full = X_tensor + actions_full

        with torch.no_grad():
            probs_gt = gt_model(X_prime_full)
            t_gt = torch.quantile(probs_gt, 1 - alpha)
            y_syn = (probs_gt >= t_gt).float()

        # Initialize an independent PyTorch weight tensor instead of relying on the gt_model architecture
        w1_weight = torch.tensor(initial_theta, dtype=torch.float32).unsqueeze(0).requires_grad_(True)
        optimizer_retrain = optim.SGD([w1_weight], lr=lr_retrain, momentum=0.9)

        for _ in range(100):
            optimizer_retrain.zero_grad()
            logits_syn = torch.matmul(X_prime_full.detach(), w1_weight.t()).squeeze()
            probs_syn = torch.sigmoid(logits_syn)
            loss_w1 = F.binary_cross_entropy(probs_syn, y_syn)
            loss_w1.backward()
            optimizer_retrain.step()

        dw1_dX = compute_ift_w1(X_prime_full, y_syn, w1_weight, exact_hessian=exact_hessian)
        X_prime_delta = X_prime_full - X_prime_full.detach()
        w1_shift = torch.einsum('ikl,kl->i', dw1_dX, X_prime_delta)
        w1_attached = w1_weight.detach() + w1_shift

        logits_full_retrained = torch.matmul(X_prime_full, w1_attached.t()).squeeze()
        probs_full_retrained = torch.sigmoid(logits_full_retrained)

        if smooth_threshold:
            t1 = SmoothedThreshold.apply(probs_full_retrained, alpha, tau)
        else:
            t1 = torch.quantile(probs_full_retrained, 1 - alpha)

        X_prime_rejected = X_rejected + actions
        logits_rejected_retrained = torch.matmul(X_prime_rejected, w1_attached.t()).squeeze()
        probs_rejected = torch.sigmoid(logits_rejected_retrained)

        validity_loss = torch.mean(torch.relu(t1 - probs_rejected))
        total_loss = validity_loss

        if eval_fn is not None:
            mean_theta = theta_param.detach().numpy()
            norm = np.linalg.norm(mean_theta)
            theta_val = (mean_theta / norm) * budget if norm > 1e-9 else mean_theta
            val = eval_fn(theta_val)
            validity_history.append(val)

        if tol > 0 and abs(prev_outer_loss - total_loss.item()) < tol:
            break
        prev_outer_loss = total_loss.item()

        total_loss.backward()
        optimizer.step()

    mean_theta = theta_param.detach().numpy()
    norm = np.linalg.norm(mean_theta)
    theta_out = (mean_theta / norm) * budget if norm > 1e-9 else mean_theta
    return theta_out, validity_history


## Baslines


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.linear_model import Ridge

def project_l2_ball(x, x0, budget):
    """
    Projects x onto an L2 ball centered at x0 with radius 'budget'.
    """
    dist = torch.norm(x - x0, p=2)
    if dist > budget:
        return x0 + (x - x0) * (budget / dist)
    return x

def get_lime_approximation(x0, model, num_samples=500, noise_std=0.05):
    """
    Approximates a PyTorch model locally around x0 using a linear surrogate (LIME-style).
    Fits a weighted Ridge regression to the model's logits.
    """
    device = x0.device
    # Generate Gaussian samples around the instance
    noise = torch.randn(num_samples, x0.shape[0], device=device) * noise_std
    X_sampled = x0.unsqueeze(0).repeat(num_samples, 1) + noise
    X_sampled = torch.cat([x0.unsqueeze(0), X_sampled], dim=0) # Include original point

    # Query the black-box model
    with torch.no_grad():
        preds = model(X_sampled).view(-1)
        # Convert probabilities to logits since ROAR assumes a linear-to-logit mapping
        preds = torch.clamp(preds, 1e-6, 1 - 1e-6)
        logits = torch.log(preds / (1 - preds))

    X_np = X_sampled.cpu().numpy()
    y_np = logits.cpu().numpy()

    # Calculate LIME weights (exponential decay based on L2 distance)
    distances = np.linalg.norm(X_np - X_np[0], axis=1)
    kernel_width = noise_std * np.sqrt(x0.shape[0])
    weights = np.exp(-(distances ** 2) / (kernel_width ** 2 + 1e-8))

    # Fit the local linear surrogate
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_np, y_np, sample_weight=weights)

    w_local = torch.tensor(ridge.coef_, dtype=torch.float32, device=device)
    b_local = torch.tensor(ridge.intercept_, dtype=torch.float32, device=device)

    return w_local, b_local

def compute_recourse_roar(x0, initial_model, delta_model, B, T=200, lr=0.01, tol=1e-4):
    """
    ROAR (Robust Algorithmic Recourse) baseline implementation.
    Automatically uses LIME for local linear approximations if the model is a Neural Network.
    """
    x = x0.clone().detach().requires_grad_(True)
    optimizer = torch.optim.Adam([x], lr=lr)

    # Check if the model is natively linear
    if hasattr(initial_model, 'linear') and hasattr(initial_model.linear, 'weight'):
        w_init = initial_model.linear.weight.detach().view(-1)
        if initial_model.linear.bias is not None:
            b_init = initial_model.linear.bias.detach().view(-1)
        else:
            b_init = torch.zeros(1, device=w_init.device)
    else:
        # LIME Fallback for Deep Neural Networks
        w_init, b_init = get_lime_approximation(x0, initial_model)

    prev_loss = float('inf')

    for t in range(T):
        optimizer.zero_grad()

        # Compute adversarial weights (worst-case model shift)
        x_sign = torch.sign(x)
        w_adv = w_init - delta_model * x_sign
        b_adv = b_init - delta_model

        logit = torch.dot(w_adv, x) + b_adv
        prob = torch.sigmoid(logit).view(-1)

        # ROAR aims to push the probability towards 1 despite the adversarial shift
        target = torch.ones_like(prob)
        current_loss = nn.MSELoss()(prob, target)

        loss_val = current_loss.item()
        if abs(prev_loss - loss_val) < tol:
            break
        prev_loss = loss_val

        current_loss.backward()
        optimizer.step()

        # Enforce budget constraint by projecting back into the L2 ball
        with torch.no_grad():
            x.data = project_l2_ball(x.data, x0, B)

    # The final recourse action is the difference between the optimized x and original x0
    action = (x - x0).detach()
    return action


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from itertools import product
from typing import Tuple

def get_rrm_theta(X: np.ndarray, y_hat_0: np.ndarray, budget_B: float, gt_scoring_fn, alpha: float, T_steps: int = 5, lr_inner: float = 0.05, max_inner_steps: int = 200, tol: float = 1e-4, exact_linear: bool = True, model_class=LogisticRegression, model_kwargs=None) -> Tuple[np.ndarray, list]:
    """
    Repeated Risk Minimization (RRM) baseline.
    If exact_linear=True, uses exact analytical weights for Logistic Regression.
    If exact_linear=False, uses local LIME approximations to handle Neural Networks and arbitrary architectures.
    """
    if model_kwargs is None:
        model_kwargs = {'penalty': None, 'fit_intercept': False, 'max_iter': 2000} if exact_linear else {}

    clf = model_class(**model_kwargs).fit(X, y_hat_0)
    validity_history = []

    # Helper to apply recourse with or without the exact linear assumption
    def apply_rrm_recourse(X_rej, clf_model):
        if exact_linear:
            w_t = clf_model.coef_[0]
            return apply_recourse_optimized(X_rej, w_t, budget_B, lr=lr_inner, max_steps=max_inner_steps, tol=tol)
        else:
            # Use LIME for each rejected individual (Neural Network Support)
            X_rec = []
            pt_wrapper = SklearnGTWrapper(clf_model)
            for x_np in X_rej:
                x0 = torch.tensor(x_np, dtype=torch.float32)
                w_l, b_l = get_lime_approximation(x0, pt_wrapper)

                x = x0.clone().detach().requires_grad_(True)
                opt = torch.optim.Adam([x], lr=lr_inner)
                prev_loss = float('inf')
                for _ in range(max_inner_steps):
                    opt.zero_grad()
                    logit = torch.dot(w_l, x) + b_l
                    prob = torch.sigmoid(logit).view(-1)
                    loss = F.binary_cross_entropy(prob, torch.ones_like(prob))

                    if abs(prev_loss - loss.item()) < tol: break
                    prev_loss = loss.item()

                    loss.backward()
                    opt.step()
                    with torch.no_grad():
                        x.data = project_l2_ball(x.data, x0, budget_B)
                X_rec.append(x.detach().numpy())
            return np.array(X_rec)

    for t in range(T_steps):
        # Shift base population according to w_t
        scores_t = clf.predict_proba(X)[:, 1]
        y_hat_t, _ = get_labels(scores_t, alpha)
        rejected_mask = (y_hat_t == 0)

        X_shifted = X.copy()
        if np.sum(rejected_mask) > 0:
            X_rejected = X[rejected_mask]
            X_recoursed = apply_rrm_recourse(X_rejected, clf)
            X_shifted[rejected_mask] = X_recoursed

            # Evaluate current step policy
            mean_action = np.mean(X_recoursed - X_rejected, axis=0)
            norm = np.linalg.norm(mean_action)
            theta_t = (mean_action / norm) * budget_B if norm > 1e-9 else mean_action
        else:
            if exact_linear:
                w_t = clf.coef_[0]
                theta_t = w_t / (np.linalg.norm(w_t) + 1e-9) * budget_B
            else:
                theta_t = np.zeros(X.shape[1])

        val, _ = evaluate_policy(X, y_hat_0, theta_t, budget_B, gt_scoring_fn, alpha, model_class, model_kwargs)
        validity_history.append(val)

        # Institution retrains on the shifted population
        clf, _ = retrain_model(X_shifted, gt_scoring_fn, alpha, model_class, model_kwargs)

    # Calculate the average direction of adaptation for the final model
    scores_final = clf.predict_proba(X)[:, 1]
    y_hat_final, _ = get_labels(scores_final, alpha)
    rejected_mask_final = (y_hat_final == 0)

    if np.sum(rejected_mask_final) > 0:
        X_rejected_final = X[rejected_mask_final]
        X_recoursed_final = apply_rrm_recourse(X_rejected_final, clf)
        mean_action = np.mean(X_recoursed_final - X_rejected_final, axis=0)
        norm = np.linalg.norm(mean_action)
        theta_rrm = (mean_action / norm) * budget_B if norm > 1e-9 else mean_action
    else:
        if exact_linear:
            w_t = clf.coef_[0]
            theta_rrm = w_t / (np.linalg.norm(w_t) + 1e-9) * budget_B
        else:
            theta_rrm = np.zeros(X.shape[1])

    # Append final validity
    val_final, _ = evaluate_policy(X, y_hat_0, theta_rrm, budget_B, gt_scoring_fn, alpha, model_class, model_kwargs)
    validity_history.append(val_final)

    return theta_rrm, validity_history

def grid_search(
    X: np.ndarray, y_hat_0: np.ndarray, budget_B: float, gt_scoring_fn, alpha: float, model_class=LogisticRegression, model_kwargs=None
) -> Tuple[np.ndarray, float, float]:
    """Finds theta that maximizes true robust validity (against t_1) via grid search."""
    best_val = -1.0
    best_theta = None
    best_t1 = 0.0

    dim = X.shape[1]
    # Safety check: Grid search is exponentially complex O(K^D).
    if dim > 3:
        return np.zeros(dim), np.nan, np.nan

    # Lower grid resolution for dimensions > 2 to prevent combinatorial explosion
    grid_points = 20 if dim <= 2 else 10
    grid = np.linspace(-3.0, 3.0, grid_points)

    for th in product(grid, repeat=dim):
        theta = np.array(th)
        if np.linalg.norm(theta) < 1e-5:
            continue

        # Use the updated generalized evaluate_policy
        val, t_1 = evaluate_policy(X, y_hat_0, theta, budget_B, gt_scoring_fn, alpha, model_class, model_kwargs)

        if val > best_val:
            best_val = val
            best_theta = theta
            best_t1 = t_1

    return best_theta, best_val, best_t1


## Experiments


## Convergence

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import pandas as pd
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone
import matplotlib.cm as cm
from sklearn.model_selection import KFold

# We need a modified version of the IFT algorithm to return the training loss (BCE of the surrogate) instead of validity
def algorithm_1_loss_tracking(X, y_hat_0, gt_model, initial_theta, max_epochs=200, budget=1.0, alpha=0.5, lr_outer=0.01, lr_inner=0.05, lr_retrain=0.5, max_inner_steps=200, tol=1e-4):
    X_tensor = torch.tensor(X, dtype=torch.float32) if not isinstance(X, torch.Tensor) else X
    current_model = gt_model

    theta_param = torch.nn.Parameter(torch.tensor(initial_theta, dtype=torch.float32))
    optimizer = torch.optim.Adam([theta_param], lr=lr_outer)

    with torch.no_grad():
        initial_probs = current_model(X_tensor)
        t0 = torch.quantile(initial_probs, 1 - alpha).item()

    rejected_mask = (initial_probs < t0).view(-1)
    X_rejected = X_tensor[rejected_mask]

    if len(X_rejected) == 0:
        return np.zeros(X_tensor.shape[1]), []

    loss_history = []

    for epoch in range(max_epochs):
        optimizer.zero_grad()
        actions = torch.zeros_like(X_rejected, requires_grad=True)
        v_a = torch.zeros_like(actions)
        beta = 0.9

        for _ in range(max_inner_steps):
            x_prime = X_rejected + actions
            logits = torch.matmul(x_prime, theta_param)
            probs = torch.sigmoid(logits)
            loss_inner = F.binary_cross_entropy(probs, torch.ones_like(probs))

            grad_a = torch.autograd.grad(loss_inner, actions, create_graph=True)[0]
            v_a = beta * v_a + (1 - beta) * grad_a
            actions = actions - lr_inner * v_a

            norms = torch.norm(actions, p=2, dim=1, keepdim=True)
            actions = torch.where(norms > budget, actions / (norms + 1e-8) * budget, actions)

        actions_full = torch.zeros_like(X_tensor)
        actions_full[rejected_mask] = actions
        X_prime_full = X_tensor + actions_full

        with torch.no_grad():
            probs_gt = gt_model(X_prime_full)
            t_gt = torch.quantile(probs_gt, 1 - alpha)
            y_syn = (probs_gt >= t_gt).float()

        w1_weight = torch.tensor(initial_theta, dtype=torch.float32).unsqueeze(0).requires_grad_(True)
        optimizer_retrain = torch.optim.SGD([w1_weight], lr=lr_retrain, momentum=0.9)

        for _ in range(50):
            optimizer_retrain.zero_grad()
            logits_syn = torch.matmul(X_prime_full.detach(), w1_weight.t()).squeeze()
            probs_syn = torch.sigmoid(logits_syn)
            loss_w1 = F.binary_cross_entropy(probs_syn, y_syn)
            loss_w1.backward()
            optimizer_retrain.step()

        dw1_dX = compute_ift_w1(X_prime_full, y_syn, w1_weight, exact_hessian=True)
        X_prime_delta = X_prime_full - X_prime_full.detach()
        w1_shift = torch.einsum('ikl,kl->i', dw1_dX, X_prime_delta)
        w1_attached = w1_weight.detach() + w1_shift

        logits_full_retrained = torch.matmul(X_prime_full, w1_attached.t()).squeeze()
        probs_full_retrained = torch.sigmoid(logits_full_retrained)
        t1 = torch.quantile(probs_full_retrained, 1 - alpha)

        X_prime_rejected = X_rejected + actions
        logits_rejected_retrained = torch.matmul(X_prime_rejected, w1_attached.t()).squeeze()
        probs_rejected = torch.sigmoid(logits_rejected_retrained)

        validity_loss = torch.mean(torch.relu(t1 - probs_rejected))

        # Track the training loss
        loss_history.append(validity_loss.item())

        validity_loss.backward()
        optimizer.step()

    mean_theta = theta_param.detach().numpy()
    norm = np.linalg.norm(mean_theta)
    theta_out = (mean_theta / norm) * budget if norm > 1e-9 else mean_theta
    return theta_out, loss_history

In [ ]:
# --- Comprehensive Convergence Visualization (Training Loss) ---

np.random.seed(42)

noise_levels = [0.0] #[0.0, 0.5]
alphas = [0.5, 0.7]#[0.3, 0.5, 0.7]
budgets = [2, 5]#[0.5, 1.0]
test_lrs = [0.01, 0.05, 0.1]
max_eps = 200
folds = 2

models_repr = {
    'Linear GT': LogisticRegression(max_iter=1000),
    'Non-Linear GT': MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=5000, random_state=42)
}

learned_models_repr = {
    'Linear Base': LogisticRegression(fit_intercept=False, max_iter=1000),
    'Non-Linear Base (MLP 16x8)': MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=5000, random_state=42),
    'Non-Linear Base (MLP 32x16)': MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=5000, random_state=42)
}

csv_data_all = []
kf = KFold(n_splits=folds, shuffle=True, random_state=42)
colors = cm.plasma(np.linspace(0, 0.9, len(test_lrs)))

for noise in noise_levels:
    print(f"\nGenerating Correlated Synthetic Data with noise {noise}...")
    X_syn_corr, f_star_syn_corr, w_star_corr = generate_correlated_nonlinear_population(n_samples=1000, rho=0.8, dim=3)
    X_syn_corr = X_syn_corr + np.random.normal(0, noise, X_syn_corr.shape)

    all_datasets = {
        f'#Synthetic (Correlated, Noise={noise})': (X_syn_corr.astype(np.float32), (f_star_syn_corr >= 0.5).astype(int)),
        #'Adult Income': (X_adult, (f_star_adult >= 0.5).astype(int)),
        #'COMPAS': (X_compas, (f_star_compas >= 0.5).astype(int)),
        'Give Me Some Credit': (X_gmsc, (f_star_gmsc >= 0.5).astype(int)),
        'Folktables (ACSIncome)': (X_folk, (f_star_folk >= 0.5).astype(int))

    }

    for data_name, (X_full, y_full) in all_datasets.items():
        for gt_model_name, gt_model_template in models_repr.items():

            gt_model = clone(gt_model_template)
            gt_model.fit(X_full, y_full)
            f_star_scores_full = gt_model.predict_proba(X_full)[:, 1]
            gt_model_pt = SklearnGTWrapper(gt_model)

            for base_model_name, base_model_template in learned_models_repr.items():
                print(f"Processing {data_name} | GT: {gt_model_name} | Base: {base_model_name}")

                # Create a figure for this specific combination of datasets/models across alphas and budgets
                fig, axes = plt.subplots(len(alphas), len(budgets), figsize=(5 * len(budgets), 4 * len(alphas)))
                fig.suptitle(f'Training Loss: {data_name}\nGT: {gt_model_name} | Base: {base_model_name}', fontsize=14)

                for a_idx, alpha in enumerate(alphas):
                    for b_idx, budget_B in enumerate(budgets):
                        ax = axes[a_idx, b_idx] if len(alphas) > 1 and len(budgets) > 1 else axes[max(a_idx, b_idx)]

                        lr_histories = {lr: [] for lr in test_lrs}

                        for train_idx, test_idx in kf.split(X_full):
                            X_train, X_test = X_full[train_idx], X_full[test_idx]
                            f_star_train = f_star_scores_full[train_idx]

                            y_train_true, _ = get_labels(f_star_train, alpha)
                            if len(np.unique(y_train_true)) < 2: continue

                            base_clf = clone(base_model_template).fit(X_train, y_train_true)
                            y_hat_0_train, _ = get_labels(base_clf.predict_proba(X_train)[:, 1], alpha)

                            if hasattr(base_clf, 'coef_'):
                                initial_theta = base_clf.coef_[0]
                            else:
                                surrogate = LogisticRegression(fit_intercept=False).fit(X_train, base_clf.predict(X_train))
                                initial_theta = surrogate.coef_[0] if len(surrogate.classes_) > 1 else np.random.randn(X_train.shape[1])

                            for lr in test_lrs:
                                _, loss_history = algorithm_1_loss_tracking(
                                    X_train, y_hat_0_train, gt_model_pt, initial_theta=initial_theta,
                                    max_epochs=max_eps, budget=budget_B, alpha=alpha,
                                    lr_outer=lr, tol=0, max_inner_steps=max_eps
                                )
                                if len(loss_history) == max_eps:
                                    lr_histories[lr].append(loss_history)

                        epochs = range(1, max_eps + 1)
                        for lr_idx, lr in enumerate(test_lrs):
                            if len(lr_histories[lr]) > 0:
                                hist_array = np.array(lr_histories[lr])
                                mean_loss = np.nanmean(hist_array, axis=0)
                                std_loss = np.nanstd(hist_array, axis=0)

                                ax.plot(epochs, mean_loss, linewidth=2, color=colors[lr_idx], label=f'LR = {lr}')
                                ax.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss, color=colors[lr_idx], alpha=0.2)

                                # Append data for saving
                                for epoch, m_loss, s_loss in zip(epochs, mean_loss, std_loss):
                                    csv_data_all.append({
                                        'Dataset': data_name,
                                        'Ground_Truth_Model': gt_model_name,
                                        'Base_Model': base_model_name,
                                        'Alpha': alpha,
                                        'Budget': budget_B,
                                        'Learning_Rate': lr,
                                        'Epoch': epoch,
                                        'Mean_Training_Loss': m_loss,
                                        'Std_Training_Loss': s_loss
                                    })

                        ax.set_title(f"Alpha={alpha}, Budget={budget_B}", fontsize=10)
                        if a_idx == len(alphas) - 1:
                            ax.set_xlabel('Epochs')
                        if b_idx == 0:
                            ax.set_ylabel('Training Loss (Validity)')
                        ax.grid(True, linestyle='--', alpha=0.5)
                        ax.legend(loc='upper right', fontsize=8)

                plt.tight_layout()
                safe_name = f"loss_{data_name.replace(' ', '_')}_{gt_model_name.replace(' ', '_')}_{base_model_name.replace(' ', '_')}.png".replace('=', '').replace(',', '')
                plt.savefig(safe_name, dpi=150)
                plt.close()

# Save all training loss results to CSV
df_csv_all = pd.DataFrame(csv_data_all)
df_csv_all.to_csv('algorithm1_training_loss_all_cv.csv', index=False)
print("\nData saved to 'algorithm1_training_loss_all_cv.csv'")

## Granular Sensitivity Sweeps


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold
import os
from sklearn.base import clone

def run_comprehensive_sweep(
    X_full, f_star_scores_full, gt_model, dataset_name, gt_name, base_models_dict,
    is_sklearn_gt=True, C_regs=None,
    inner_steps=20, outer_epochs=50, lr=0.1,
    alphas=None, budgets=None, folds=2, seed=42
):
    if alphas is None:
        alphas = [0.1, 0.3, 0.5, 0.7, 0.9]
    if budgets is None:
        budgets = [0.5, 1.0, 1.5, 2.0]
    if C_regs is None:
        C_regs = [0.001]

    gt_model_pt = SklearnGTWrapper(gt_model) if is_sklearn_gt else gt_model
    gt_scoring_fn = lambda X: gt_model.predict_proba(X)[:, 1] if is_sklearn_gt else (lambda X: gt_model(torch.tensor(X, dtype=torch.float32)).detach().numpy())

    np.random.seed(seed)
    kf = KFold(n_splits=folds, shuffle=True, random_state=seed)
    dim = X_full.shape[1]
    csv_rows = []

    for base_name, base_template in base_models_dict.items():
        for C_reg in C_regs:
            print(f"\nEvaluating GT: {gt_name} | Base: {base_name} | C_reg: {C_reg} on {dataset_name}")
            for B in budgets:
                for alpha in alphas:
                    fold_vals = {'Naive': [], 'ROAR': [], 'RRM': [], 'Alg 1 (IFT)': []}

                    for train_idx, test_idx in kf.split(X_full):
                        X, X_test = X_full[train_idx], X_full[test_idx]
                        f_star_scores = f_star_scores_full[train_idx]

                        y_true, _ = get_labels(f_star_scores, alpha)
                        if len(np.unique(y_true)) < 2: continue

                        # Train Base Model with specific Regularizer
                        if isinstance(base_template, LogisticRegression):
                            base_template.set_params(C=C_reg)
                        elif isinstance(base_template, MLPClassifier):
                            base_template.set_params(alpha=1.0 / C_reg)

                        base_clf = clone(base_template).fit(X, y_true)

                        # Get predictions for base model
                        y_hat_0_train, _ = get_labels(base_clf.predict_proba(X)[:, 1], alpha)
                        y_hat_0_test, _ = get_labels(base_clf.predict_proba(X_test)[:, 1], alpha)

                        if np.sum(y_hat_0_test == 0) == 0: continue

                        # If base model is non-linear, we need a linear surrogate to get a single direction (theta)
                        if hasattr(base_clf, 'coef_'):
                            theta_naive = base_clf.coef_[0]
                            model_class = type(base_clf)
                            model_kwargs = base_clf.get_params()
                        else:
                            surrogate = LogisticRegression(C=C_reg, fit_intercept=False).fit(X, y_hat_0_train)
                            theta_naive = surrogate.coef_[0] if len(surrogate.classes_) > 1 else np.random.randn(dim)
                            model_class = type(base_clf)
                            model_kwargs = base_clf.get_params()

                        # --- Naive ---
                        val_naive, _ = evaluate_policy(X_test, y_hat_0_test, theta_naive, B, gt_scoring_fn, alpha, model_class, model_kwargs)
                        fold_vals['Naive'].append(val_naive)

                        # --- ROAR ---
                        initial_pt_model = LogisticRegressionPT(input_dim=dim)
                        with torch.no_grad():
                            initial_pt_model.linear.weight.copy_(torch.tensor(np.array([theta_naive]), dtype=torch.float32))

                        X_rejected = X[y_hat_0_train == 0]
                        if len(X_rejected) > 0:
                            X_rejected_sub = X_rejected[np.random.choice(len(X_rejected), min(len(X_rejected), 30), replace=False)]
                            roar_actions = [compute_recourse_roar(torch.tensor(x, dtype=torch.float32), initial_pt_model, delta_model=0.1, B=B, T=inner_steps, lr=lr).numpy() for x in X_rejected_sub]
                            mean_roar = np.mean(roar_actions, axis=0)
                            norm_roar = np.linalg.norm(mean_roar)
                            theta_roar = (mean_roar / norm_roar) * B if norm_roar > 1e-9 else mean_roar
                        else:
                            theta_roar = np.zeros(dim)
                        val_roar, _ = evaluate_policy(X_test, y_hat_0_test, theta_roar, B, gt_scoring_fn, alpha, model_class, model_kwargs)
                        fold_vals['ROAR'].append(val_roar)

                        # --- RRM ---
                        theta_rrm, _ = get_rrm_theta(X, y_hat_0_train, B, gt_scoring_fn, alpha, T_steps=10, max_inner_steps=inner_steps, exact_linear=hasattr(base_clf, 'coef_'), model_class=model_class, model_kwargs=model_kwargs)
                        val_rrm, _ = evaluate_policy(X_test, y_hat_0_test, theta_rrm, B, gt_scoring_fn, alpha, model_class, model_kwargs)
                        fold_vals['RRM'].append(val_rrm)

                        # --- Alg 1 (IFT) ---
                        theta_ift, _ = algorithm_1_optimized_ce_gen(
                            X, y_hat_0_train, gt_model_pt, initial_theta=theta_naive,
                            max_epochs=outer_epochs*4, max_inner_steps=inner_steps*2,
                            budget=B, alpha=alpha, lr_outer=lr, lr_inner=lr, tol=1e-3
                        )
                        val_ift, _ = evaluate_policy(X_test, y_hat_0_test, theta_ift, B, gt_scoring_fn, alpha, model_class, model_kwargs)
                        fold_vals['Alg 1 (IFT)'].append(val_ift)

                    for m in fold_vals.keys():
                        mean_val = np.nanmean(fold_vals[m]) if len(fold_vals[m]) > 0 else np.nan
                        std_val = np.nanstd(fold_vals[m]) if len(fold_vals[m]) > 0 else np.nan
                        csv_rows.append({
                            'Dataset': dataset_name, 'GT_Model': gt_name, 'Base_Model': base_name,
                            'C_reg': C_reg, 'Method': m, 'Budget': B, 'Alpha': alpha, 
                            'Validity': mean_val, 'Validity_Std': std_val,
                            'Fold_Values': fold_vals[m].copy() 
                        })

    # Save local results to CSV
    if len(csv_rows) > 0:
        df_res = pd.DataFrame(csv_rows)
        safe_dname = dataset_name.replace(' ', '_').lower()
        safe_gtname = gt_name.replace(' ', '_').lower()
        fname = f"comprehensive_sweep_{safe_dname}_{safe_gtname}.csv"
        df_res['Fold_Values'] = df_res['Fold_Values'].apply(lambda x: str(x))
        df_res.to_csv(fname, index=False)
        print(f"\nSaved local sweep results to '{fname}'")

    return csv_rows

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

# --- Experiment Parameters ---
QUICK_RUN = True  # Set to False to run the full paper experiments
N_SAMPLES = 1000 if QUICK_RUN else 2000
FOLDS = 3 if QUICK_RUN else 5
BUDGETS = [0.5, 1.0, 2.0, 3.0, 5]
ALPHAS = [0.2, 0.4, 0.6, 0.8]
C_REGS = [0.001, 0.01, 0.1, 1]
NOISE_LEVELS = [0.0, 0.5, 1, 2]
INNER_STEPS = 20
OUTER_EPOCHS = 50
LR = 0.1
SEED = 100

print("Preparing Datasets...")
datasets = {}

# 1. Synthetic Data with Different Noise Levels
for noise in NOISE_LEVELS:
    X_syn, f_star_syn, _ = generate_correlated_nonlinear_population(n_samples=N_SAMPLES, rho=0.8, dim=3)
    X_syn = X_syn + np.random.normal(0, noise, X_syn.shape)
    datasets[f'Synthetic (Noise={noise})'] = (X_syn.astype(np.float32), f_star_syn)

# 2. Adult Income
X_adult, f_star_adult, _, gt_adult = prepare_adult_data(n_samples=N_SAMPLES, seed=42)
datasets['Adult Income'] = (X_adult, f_star_adult)

# 3. German Credit
X_credit, f_star_credit, _, gt_credit = prepare_credit_data(n_samples=N_SAMPLES, seed=42)
datasets['German Credit'] = (X_credit, f_star_credit)

# 4. COMPAS
X_compas, f_star_compas, _, gt_compas = prepare_compas_data(n_samples=N_SAMPLES, seed=42)
datasets['COMPAS'] = (X_compas, f_star_compas)

# 5. Give Me Some Credit
X_gmsc, f_star_gmsc, _, gt_gmsc = prepare_gmsc_data(n_samples=N_SAMPLES, seed=42)
datasets['Give Me Some Credit'] = (X_gmsc, f_star_gmsc)

# 6. Folktables (ACSIncome)
X_folk, f_star_folk, _, gt_folk = prepare_folktables_data(n_samples=N_SAMPLES, seed=42)
datasets['Folktables (ACSIncome)'] = (X_folk, f_star_folk)

# GT Models
gt_models_dict = {
    'Linear GT': LogisticRegression(max_iter=5000),
    'Non-Linear GT (MLP 100x50)': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=5000, random_state=SEED)
}

# Base Models
base_models_dict = {
    'Linear Base': LogisticRegression(fit_intercept=False, max_iter=5000),
    'Non-Linear Base (MLP 16x8)': MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=5000, random_state=SEED)
}

all_comprehensive_results = []

for d_name, (X_data, f_star_scores) in datasets.items():
    for gt_name, gt_template in gt_models_dict.items():
        print(f"\n{'='*50}")
        print(f"Dataset: {d_name} | Training GT: {gt_name}")
        print(f"{'='*50}")

        # Train the actual GT to compute probabilities
        y_data = (f_star_scores >= 0.5).astype(int)
        if len(np.unique(y_data)) < 2:
             print(f"Skipping {d_name} due to single class in GT generation.")
             continue

        gt_model = clone(gt_template).fit(X_data, y_data)
        f_star_recalc = gt_model.predict_proba(X_data)[:, 1]

        res = run_comprehensive_sweep(
            X_data, f_star_recalc, gt_model, d_name, gt_name, base_models_dict, is_sklearn_gt=True,
            C_regs=C_REGS, inner_steps=INNER_STEPS, outer_epochs=OUTER_EPOCHS, lr=LR, alphas=ALPHAS,
            budgets=BUDGETS, folds=FOLDS
        )
        all_comprehensive_results.extend(res)

df_comprehensive = pd.DataFrame(all_comprehensive_results)
df_comprehensive.to_csv('comprehensive_sensitivity_sweep_results.csv', index=False)
print("\nAll comprehensive results saved to 'comprehensive_sensitivity_sweep_results.csv'")
display(df_comprehensive.head(10))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configure global plotting aesthetics
sns.set_theme(style="whitegrid")
custom_dashes = {'Naive': (4, 2), 'ROAR': (1, 0), 'RRM': (1, 0), 'Alg 1 (IFT)': (1, 0)}
custom_markers = {'Naive': 'o', 'ROAR': 'X', 'RRM': 's', 'Alg 1 (IFT)': 'D'}

def plot_and_save_metric(df, x_col, fixed_col, fixed_val, gt_model, base_model, c_reg, title_prefix, filename):
    # Filter data for the specific model combination and fixed parameter
    plot_df = df[(df['GT_Model'] == gt_model) & 
                 (df['Base_Model'] == base_model) & 
                 (df['C_reg'] == c_reg)]
    
    if fixed_col:
        plot_df = plot_df[plot_df[fixed_col] == fixed_val]

    datasets = plot_df['Dataset'].unique()
    n_ds = len(datasets)
    
    if n_ds == 0:
        return # Skip if no data is found for this combination

    # Removed sharey=True so each plot gets its own y-axis ticks
    fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 4.5), sharey=False)
    if n_ds == 1: axes = [axes]

    fig.suptitle(f'{title_prefix}\nGT: {gt_model} | Base: {base_model} | C_reg: {c_reg}', fontsize=14, fontweight='bold', y=1.1)

    for ax, ds in zip(axes, datasets):
        subset = plot_df[plot_df['Dataset'] == ds]
        
        # Plotting the pre-calculated fold averages ('Validity')
        sns.lineplot(
            data=subset, x=x_col, y='Validity', hue='Method', style='Method',
            markers=custom_markers, dashes=custom_dashes, ax=ax, linewidth=2.5,
            markersize=8, errorbar=None
        )
        ax.set_title(ds, fontsize=13)
        ax.set_xlabel(x_col.replace('_', ' '), fontsize=12)
        ax.set_ylabel('Robust Validity', fontsize=12) # Applied to all subplots
        ax.set_ylim(0, 1.05)

        # Removed the logic that hides the legend for ax != axes[0]

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved {filename}")
    plt.show()

# 1. Plot Budget and Alpha from Comprehensive Sweep
try:
    df_comp = pd.read_csv('comprehensive_sensitivity_sweep_results.csv')
    print("Loaded comprehensive_sensitivity_sweep_results.csv")
    
    gt_models = df_comp['GT_Model'].dropna().unique()
    base_models = df_comp['Base_Model'].dropna().unique()
    c_regs = df_comp['C_reg'].dropna().unique()

    # Plot Effect of Budget for ALL Alphas
    unique_alphas = sorted(df_comp['Alpha'].dropna().unique())
    for gt in gt_models:
        for base in base_models:
            for c in c_regs:
                for alpha_val in unique_alphas:
                    safe_gt = gt.replace(' ', '_').replace('(', '').replace(')', '').lower()
                    safe_base = base.replace(' ', '_').replace('(', '').replace(')', '').lower()
                    fname = f'effect_of_budget_alpha_{alpha_val}_{safe_gt}_{safe_base}_creg_{c}.png'
                    
                    plot_and_save_metric(
                        df=df_comp, x_col='Budget', fixed_col='Alpha', fixed_val=alpha_val,
                        gt_model=gt, base_model=base, c_reg=c,
                        title_prefix=f'Effect of Recourse Budget (Alpha = {alpha_val})',
                        filename=fname
                    )

    # Plot Effect of Alpha for ALL Budgets
    unique_budgets = sorted(df_comp['Budget'].dropna().unique())
    for gt in gt_models:
        for base in base_models:
            for c in c_regs:
                for budget_val in unique_budgets:
                    safe_gt = gt.replace(' ', '_').replace('(', '').replace(')', '').lower()
                    safe_base = base.replace(' ', '_').replace('(', '').replace(')', '').lower()
                    fname = f'effect_of_alpha_budget_{budget_val}_{safe_gt}_{safe_base}_creg_{c}.png'
                    
                    plot_and_save_metric(
                        df=df_comp, x_col='Alpha', fixed_col='Budget', fixed_val=budget_val,
                        gt_model=gt, base_model=base, c_reg=c,
                        title_prefix=f'Effect of Rejection Rate / Alpha (Budget = {budget_val})',
                        filename=fname
                    )
except FileNotFoundError:
    print("Could not find 'comprehensive_sensitivity_sweep_results.csv'. Make sure the sweep completed and saved the file.")

# 2. Plot Noise from Noise Sweep (If available)
# Adjusting this script might also be needed if the noise sweep contains different models/c_regs
try:
    df_noise = pd.read_csv('noise_sweep_all_results.csv')
    print("\nLoaded noise_sweep_all_results.csv")
    # Example fallback if noise sweep was not formatted with GT/Base Model combinations:
    # Feel free to update this logic identically to above if df_noise has the same columns.
except FileNotFoundError:
    print("Could not find 'noise_sweep_all_results.csv'. Make sure the sweep completed and saved the file.")